<a href="https://colab.research.google.com/github/OjasKhandelwal/IMDB-Sentiment-Classifier/blob/main/IMDB_Binary_Sentiment_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##SETUP

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate gradio

In [ ]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, f1_score

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

##LOAD THE DATASET

In [ ]:
dataset = load_dataset("imdb")

In [ ]:
print(dataset)

In [ ]:
sample = dataset["train"][0]
print("Label:", sample["label"], "(0 = negative, 1 = positive)")
print("\nReview:", sample["text"][:500], "...")

In [ ]:
TRAIN_SIZE = 2000
TEST_SIZE = 500

small_train = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
small_test = dataset["train"].shuffle(seed=SEED).select(range(TEST_SIZE))

# split the training data into train + validation
split = small_train.train_test_split(test_size=0.1, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Train size:", len(small_train))
print("Val size:", len(val_ds))
print("Test size:", len(small_test))

##TOKENIZE THE TEXT

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# tokenize all three sets
train_tokenized = train_ds.map(tokenize, batched=True)
val_tokenized = val_ds.map(tokenize, batched=True)
test_tokenized = small_test.map(tokenize, batched=True)

##LOAD THE PRE-TRAINED BERT FOR CLASSIFICATION

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

##METRICS & DATA COLLATOR

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

##SETUP THE TRAINER

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_imdb_output",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

##FINE-TUNING THE MODEL

In [ ]:
trainer.train()

##EVALUATE ON TEST SET

In [ ]:
final_results = trainer.evaluate(eval_dataset=test_tokenized)
print("Final results:")
for k, v in final_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

##PREDICT SENTIMENT ON NEW REVIEWS

In [ ]:
id2label = {0: "negative", 1: "positive"}

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "label": id2label[pred_id],
        "confidence": round(probs[pred_id].item(), 4),
    }

# Try it out
reviews = [
    "This movie was absolutely fantastic. Brilliant acting and a gripping story.",
    "I hated this film. It was boring, too long, and badly written.",
    "It was okay — some good moments, but overall forgettable.",
]

for r in reviews:
    print(predict_sentiment(r), "->", r[:60], "...")

##SAVING THE MODEL & TOKENIZER

In [ ]:
SAVE_DIR = "./imdb_bert"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer to:", SAVE_DIR)

##Gradio UI

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ---- Reload from saved directory ----
SAVE_DIR = "./imdb_bert"

reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
reloaded_model.to(device)
reloaded_model.eval()

print("Model and tokenizer reloaded on:", device)

# ---- Inference function ----
id2label = {0: "negative", 1: "positive"}
MAX_LENGTH = 256

def predict_sentiment(text):
    inputs = reloaded_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)

    with torch.no_grad():
        logits = reloaded_model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()
    return id2label[pred_id], probs[pred_id].item()


# ---- Gradio UI ----
def classify_review(review):
    if not review.strip():
        return "Please enter a review."
    label, confidence = predict_sentiment(review)
    emoji = "🙂" if label == "positive" else "🙁"
    return f"{emoji} **{label.upper()}** (confidence: {confidence:.2%})"

demo = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type a movie review here...",
        label="Movie Review"
    ),
    outputs=gr.Markdown(label="Prediction"),
    title="IMDB Sentiment Classifier",
    description="Fine-tuned BERT predicting whether a movie review is positive or negative.",
    flagging_mode="never",
    examples=[
        ["This movie was absolutely fantastic. Brilliant acting and a gripping story."],
        ["I hated this film. It was boring, too long, and badly written."],
        ["It was okay — some good moments, but overall forgettable."],
    ],
)

demo.launch(share=True)